# VLDB First Upgrade: Selected Adaptive Validation
CPU-only Stage 3 campaign over three regimes selected from measured regional screening results.

In [ ]:
from pathlib import Path
import shutil
import zipfile

input_root = Path('/kaggle/input')
root = Path('/kaggle/working/SIGMOD')
if root.exists():
    shutil.rmtree(root)
root.mkdir(parents=True)
bundles = list(input_root.rglob('kaggle_upload_dynamic_selected.zip'))
if len(bundles) == 1:
    with zipfile.ZipFile(bundles[0]) as archive:
        archive.extractall(root)
elif len(bundles) == 0:
    configs = list(input_root.rglob('experiments/configs/dynamic/selected_high.json'))
    assert len(configs) == 1, 'Expected one unpacked selected-adaptive source tree'
    source_root = configs[0].parents[3]
    for item in source_root.iterdir():
        target = root / item.name
        shutil.copytree(item, target) if item.is_dir() else shutil.copy2(item, target)
else:
    raise AssertionError(f'Expected at most one selected-adaptive ZIP, found {len(bundles)}')
required = [root / 'experiments/configs/dynamic/selected_zero.json', root / 'experiments/configs/dynamic/selected_borderline.json', root / 'experiments/configs/dynamic/selected_high.json', root / 'experiments/configs/dynamic/oracle_zero.json']
assert all(p.is_file() for p in required), [str(p) for p in required if not p.is_file()]
print(root)
print('Bundle extraction and measured-oracle checks passed.')


In [ ]:
!uname -a
!lscpu
!free -h
!g++ --version
!nvidia-smi || true


In [ ]:
!bash {root}/scripts/linux/build_release.sh --execute
!{root}/build/prepared/test_v2


In [ ]:
import os
import subprocess
os.environ['CABTREE_CXXFLAGS'] = '-std=c++11 -O3 -march=native -DNDEBUG'
oracle_configs = ['oracle_zero.json', 'oracle_borderline.json', 'oracle_high.json']
configs = ['selected_zero.json', 'selected_borderline.json', 'selected_high.json', 'perfect_zero.json', 'perfect_borderline.json', 'perfect_high.json']
raw_root = root / 'results/raw/kaggle'
def config_out(name):
    return raw_root / Path(name).stem
pre_oracle_configs = oracle_configs + configs[:3]
expected = [30, 30, 30, 450, 450, 450]
for name, count in zip(pre_oracle_configs, expected):
    cmd = ['python3', str(root / 'experiments/campaign.py'), '--config', str(root / 'experiments/configs/dynamic' / name), '--binary', str(root / 'build/prepared/bench_unified'), '--out-dir', str(config_out(name))]
    check = subprocess.run(cmd, check=True, capture_output=True, text=True)
    assert f'planned runs: {count}' in check.stdout, (name, check.stdout[-500:])
    print(name, count)
print('Pre-oracle dry-run total: 1440')


In [ ]:
# Calibrate the phase oracle with this exact binary and machine.
for name in oracle_configs:
    print('CALIBRATING', name, flush=True)
    subprocess.run(['python3', str(root / 'experiments/campaign.py'), '--config', str(root / 'experiments/configs/dynamic' / name), '--binary', str(root / 'build/prepared/bench_unified'), '--out-dir', str(config_out(name)), '--execute'], check=True)


In [ ]:
oracle_csv = root / 'results/processed/dynamic_selected_phase_oracle.csv'
subprocess.run(['python3', str(root / 'analysis/phase_oracle.py'), '--raw', str(raw_root), '--family', 'dynamic_selected_oracle', '--output', str(oracle_csv)], check=True)
assert oracle_csv.is_file()
print(oracle_csv)
for name in configs[3:]:
    cmd = ['python3', str(root / 'experiments/campaign.py'), '--config', str(root / 'experiments/configs/dynamic' / name), '--binary', str(root / 'build/prepared/bench_unified'), '--out-dir', str(config_out(name))]
    check = subprocess.run(cmd, check=True, capture_output=True, text=True)
    assert 'planned runs: 100' in check.stdout, (name, check.stdout[-500:])
    print(name, 100)
print('Validated total campaign runs: 1740')
# Checkpointed evaluation: completed CSVs are skipped safely on notebook resume.
for name in configs:
    print('EVALUATING', name, flush=True)
    subprocess.run(['python3', str(root / 'experiments/campaign.py'), '--config', str(root / 'experiments/configs/dynamic' / name), '--binary', str(root / 'build/prepared/bench_unified'), '--out-dir', str(config_out(name)), '--phase-oracle-csv', str(oracle_csv), '--execute'], check=True)


In [ ]:
!python3 {root}/analysis/validate_pairs.py --raw {root}/results/raw/kaggle
!python3 {root}/analysis/aggregate.py --raw {root}/results/raw/kaggle --output {root}/results/processed/dynamic_selected_summary.csv
from pathlib import Path
raw = root / 'results/raw/kaggle'
main_csvs = [p for p in raw.rglob('*.csv') if not p.name.endswith('.phases.csv') and 'partial' not in p.name]
phase_csvs = list(raw.rglob('*.phases.csv'))
failures = list(raw.rglob('*.failed.json'))
partials = list(raw.rglob('*.partial.csv'))
print('main CSVs:', len(main_csvs), 'phase CSVs:', len(phase_csvs), 'failures:', len(failures), 'partials:', len(partials))
assert len(main_csvs) == 1740 and len(phase_csvs) == 1740 and not failures and not partials


In [ ]:
# Final cell: preserve complete results and a resumable campaign bundle.
results_zip = shutil.make_archive('/kaggle/working/results', 'zip', root / 'results')
resume_root = Path('/kaggle/working/resume_bundle')
if resume_root.exists():
    shutil.rmtree(resume_root)
shutil.copytree(root / 'results', resume_root / 'results')
shutil.copytree(root / 'experiments/configs/dynamic', resume_root / 'configs')
shutil.copy2(root / 'kaggle/kaggle_dynamic_selected.ipynb', resume_root / 'kaggle_dynamic_selected.ipynb')
resume_zip = shutil.make_archive('/kaggle/working/resume', 'zip', resume_root)
print(results_zip)
print(resume_zip)
